In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = sns.load_dataset('iris')

In [3]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [4]:
df["species"].value_counts()

,count
species,
setosa,50
versicolor,50
virginica,50


In [5]:
from sklearn.model_selection import train_test_split
X = df.drop('species', axis = 1)
y = df['species']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [8]:
models = {
    "knn": KNeighborsClassifier(),
    "tree": DecisionTreeClassifier(),
    "log": LogisticRegression(),
    "svm": SVC()
}

#GridSearchCV

In [9]:
from sklearn.model_selection import GridSearchCV

In [10]:
params_grid = {
    "knn": {
        "n_neighbors": np.arange(1, 20),
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan"]
    },
    "tree": {
        "max_depth": np.arange(1, 15),
        "criterion": ["gini", "entropy"],
        "min_samples_split": [2, 5, 10]
    },
    "svm": {
        "C": np.logspace(-3, 2, 6),
        "gamma": np.logspace(-3, 2, 6),
        "kernel": ["linear", "rbf"]
    },
    "log": {
        "C": np.logspace(-3, 2, 6),
        "penalty": ["l2"],
        "solver": ["lbfgs", "liblinear"]
    }
}


for name, model in models.items():
    grid = GridSearchCV(model, params_grid[name], cv=5)
    grid.fit(X, y)

    # 1. Round and format the score
    score = round(grid.best_score_, 2)

    print(f"{name.upper()}:")
    print(f"  Best Score: {score}")
    print(f"  Best Params: {grid.best_params_}")
    print("-" * 30)

KNN:
  Best Score: 0.99
  Best Params: {'metric': 'euclidean', 'n_neighbors': np.int64(10), 'weights': 'distance'}
------------------------------
TREE:
  Best Score: 0.97
  Best Params: {'criterion': 'gini', 'max_depth': np.int64(3), 'min_samples_split': 2}
------------------------------
LOG:
  Best Score: 0.98
  Best Params: {'C': np.float64(10.0), 'penalty': 'l2', 'solver': 'lbfgs'}
------------------------------
SVM:
  Best Score: 0.98
  Best Params: {'C': np.float64(1.0), 'gamma': np.float64(0.001), 'kernel': 'linear'}
------------------------------


# RandomSearchCV

In [11]:
from sklearn.model_selection import RandomizedSearchCV

for name, model in models.items():
  random_grid = RandomizedSearchCV(model, params_grid[name], cv=5)
  random_grid.fit(X, y)
  score = round(random_grid.best_score_, 2)

  print(f"{name.upper()}:")
  print(f"  Best Score: {score}")
  print(f"  Best Params: {random_grid.best_params_}")
  print("-" * 30)

KNN:
  Best Score: 0.99
  Best Params: {'weights': 'distance', 'n_neighbors': np.int64(10), 'metric': 'euclidean'}
------------------------------
TREE:
  Best Score: 0.97
  Best Params: {'min_samples_split': 10, 'max_depth': np.int64(4), 'criterion': 'gini'}
------------------------------
LOG:
  Best Score: 0.98
  Best Params: {'solver': 'lbfgs', 'penalty': 'l2', 'C': np.float64(100.0)}
------------------------------
SVM:
  Best Score: 0.97
  Best Params: {'kernel': 'linear', 'gamma': np.float64(100.0), 'C': np.float64(0.1)}
------------------------------


#Ensemble Methods

#Stacking

In [12]:
data = df.copy()

In [13]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

In [14]:
y_encoded = encoder.fit_transform(y)

In [15]:
X_train_encode, X_test_encode, y_train_encode, y_test_encode = train_test_split(X, y_encoded, random_state=42, test_size=0.2)

In [16]:
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import classification_report

In [17]:
base_learners = {
    "tree": DecisionTreeClassifier(),
    "log": LogisticRegression(),
    "svm": SVC()
}

meta_learner = KNeighborsClassifier(n_neighbors=5)

In [18]:
stacking_clf = StackingClassifier(
    estimators=[(name, model) for name, model in base_learners.items()],
    final_estimator=meta_learner,
    cv=5
)
stacking_clf.fit(X_train_encode, y_train_encode)

StackingClassifier(cv=5,
                   estimators=[('tree', DecisionTreeClassifier()),
                               ('log', LogisticRegression()), ('svm', SVC())],
                   final_estimator=KNeighborsClassifier())

In [19]:
y_pred_stk = stacking_clf.predict(X_test_encode)

In [20]:
accuracy = accuracy_score(y_test_encode, y_pred_stk)
print(f"Accuracy: {accuracy}")

Accuracy: 1.0


Since the dataset is too small, it is overfitting on the model.

#Bagging

In [21]:
from sklearn.ensemble import RandomForestClassifier

In [22]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

In [23]:
rf_model.fit(X_train_encode, y_train_encode)

RandomForestClassifier(max_depth=5, random_state=42)

In [24]:
y_pred_rf = rf_model.predict(X_test_encode)

In [25]:
accuracy = accuracy_score(y_test_encode, y_pred_rf)
print(f"Accuracy: {accuracy}")

Accuracy: 1.0


In [26]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

In [28]:
ada_model = AdaBoostClassifier(
    n_estimators=50,
    random_state=42
)

In [29]:
ada_model.fit(X_train, y_train)

AdaBoostClassifier(random_state=42)

In [30]:
y_pred_ada = ada_model.predict(X_test)

In [31]:
print(f"Accuracy: {accuracy_score(y_test, y_pred_ada)}")

Accuracy: 0.9333333333333333


In [32]:
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

In [33]:
gb_model.fit(X_train, y_train)

GradientBoostingClassifier(random_state=42)

In [34]:
y_pred_gb = gb_model.predict(X_test)

In [35]:
print(f"Accuracy: {accuracy_score(y_test, y_pred_gb)}")

Accuracy: 1.0


In [39]:
xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

In [41]:
xgb.fit(X_train_encode, y_train_encode)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [42]:
y_pred_xgb = xgb.predict(X_test_encode)

In [44]:
print(f"accuracy: {accuracy_score(y_test_encode, y_pred_xgb)}")

accuracy: 1.0
